In [1]:
# Pass marks, set before the run.
# The estimated cost of every LLM call on the full test set, saved replies included: about $5 per 1,000
# excerpts on the sample (notebook 15), so about $65.
MAX_LLM_COST_USD = 80.0
MAX_MALFORMED_SHARE = 0.05  # replies that are not the JSON asked for, per method
# The full run is the sample's run on more excerpts: on the sample's excerpts, every method's labels must
# give the exact-label scores notebook 15 saved, and the combined method's macro F1 must be the full-test
# figure notebook 15 saved, to within this much. Any difference raises.
SAME_WITHIN = 1e-9

# 17 · Every HumSet 2.0.0 method on the full test set

The HumSet tables first scored every method on 1,500 test excerpts, 500 in each of English, French and Spanish,
because each LLM method pays per excerpt (notebooks 14, 15 and 16). HumBERT's paper reports macro F1 on all of
HumSet 2.0.0's test excerpts, so a sample cannot be set beside its figures. This notebook scores every method on
all 12,896 of them, both ways the tables show: exact labels and tier by tier, as notebook 15, and HumBERT's
macro F1 per published group, as notebook 16. HumSet 1.0.0, which the paper does not report on, keeps its
sample.

Each method is run exactly as on the sample, with the same settings, tuning items and cut-offs, so the sample's
own excerpts get the same labels and are checked against the saved sample scores. The LLM methods are run on
every test excerpt; replies already saved are reused. The trained, fine-tuned and combined methods read the
rankings their notebooks saved for every test excerpt (notebooks 08, 10, 13 and 15).

The cut-offs and per-label thresholds of every ranked method are chosen on the same 4,000 development excerpts,
because the combined method trains on the other 6,000. HumBERT's own trainer chooses its thresholds its own way
on every validation excerpt; notebook 13 scores the recipe that way, and this notebook also scores it with our
thresholds chosen on every development excerpt, to show how much each difference moves it.

HumSet 2.0.0 tags some excerpts with a pillar but none of its subpillars; such a pillar is a label of its own
(`datasets.humset_labels`).

In [2]:
import numpy as np
import pandas as pd

from tiered_taxonomy import benchmarks as bm
from tiered_taxonomy import experiment as ex
from tiered_taxonomy import like_for_like as lf
from tiered_taxonomy import new_technique as nt
from tiered_taxonomy import runs
from tiered_taxonomy import signals as sg
from tiered_taxonomy.config import data_root, load_config
from tiered_taxonomy.methods import combined as cb
from tiered_taxonomy.methods.trained import choose_cut

root = data_root()
config = load_config()
seed = config["seed"]
words = config["text"]["input_words"]
limits, thresholds = config["trained"]["cut_limits"], config["trained"]["cut_thresholds"]
settings = cb.CombinedSettings(per_source=config["combined"]["per_source"], seed=seed)
plan = config["new_technique"]["humset"]
SAMPLE_SCORES = ex.load_results("15_humset_v2")
STACK = SAMPLE_SCORES["choice"]["chosen"]
FULL_TEST_COMBINED = SAMPLE_SCORES["full_test"]["scores"]
SOURCES = list(SAMPLE_SCORES["ranked"])
data = ex.load_humset(root, ex.HUMSET_V2)

/Users/matthewharris/git/nvidia-autoontology-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## The LLM methods, on every test excerpt

As in notebook 15, which runs them as notebooks 03 to 06 and 09 did on version 1.0.0. The two recipes from
Kähler et al. choose their cut on the same development excerpts as there. The cost is checked as it goes.

In [3]:
def run_llm(data):
    """Run every LLM method on the version's test set; return each one's labels and what it cost."""
    tuning = ex.pilot_subset(data.dev, config["pipeline"]["tuning_items"], seed)
    bench = runs.make_bench(data, root, config)
    runs.load_cards(bench)
    runs.embed_inputs(bench, data.test)
    spent = 0.0
    made = {}

    def keep(name, labels, run, charged):
        nonlocal spent
        spent += charged
        share = run["malformed"] / run["calls"] if run["calls"] else 0.0
        print(f"{data.name} {name}: ${run['cost_usd']:.2f}, malformed {share:.1%}; spent ${spent:.2f}")
        if share > MAX_MALFORMED_SHARE:
            raise RuntimeError(f"{name}: {share:.1%} of replies were malformed")
        if spent > MAX_LLM_COST_USD:
            raise RuntimeError(f"LLM calls have cost ${spent:.2f}, over the limit")
        if labels is not None:
            made[name] = {
                "labels": labels,
                "cost_per_1000": round(run["cost_usd"] / run["items"] * 1000, 2),
                "calls_per_item": round(run["calls"] / run["items"], 2),
            }

    ids = [item.id for item in data.test]
    runs.tune_pipeline(bench, tuning)
    for method in (
        "pipeline",
        "select_all",
        "funnel_names",
        "funnel_cards",
        "walk_strict",
        "walk_branching",
        "walk_anchored",
    ):
        outcome = runs.run_and_score(bench, method, data.test)
        labels = ex.load_predictions(root, data.name, method, ids)
        keep(method, labels, outcome["run"], outcome["run"]["cost_usd"])
    for method, outcome in runs.evidence_runs(bench, "funnel_cards", data.test).items():
        if method != "funnel_cards":
            labels = ex.load_predictions(root, data.name, method, ids)
            keep(method, labels, outcome["run"], outcome["run"]["cost_usd"])
    recipe_bench = runs.make_bench(data, root, config)
    recipe_bench.workers = config["paper_models"]["workers"]
    tools = runs.make_recipe_tools(recipe_bench)
    for recipe in ("llm_ensemble", "ki_fsprompt"):
        dev_ranked, dev_record = runs.run_recipe(recipe_bench, tools, recipe, "dev", tuning)
        keep(f"{recipe} tuning", None, dev_record.__dict__, dev_record.cost_usd)
        cut, _ = choose_cut(ex.gold_exact(tuning), dev_ranked, limits, thresholds)
        ranked, record = runs.run_recipe(recipe_bench, tools, recipe, "test", data.test)
        labels = [cut.apply(ranking) for ranking in ranked]
        ex.save_predictions(root, data.name, recipe, ids, labels)
        keep(recipe, labels, record.__dict__, record.cost_usd)
    return made, spent


made, spent = run_llm(data)
print(f"LLM calls: ${spent:.2f}, saved replies included")

humset_v2 pipeline: $4.27, malformed 0.0%; spent $4.27


humset_v2 select_all: $1.88, malformed 0.0%; spent $6.15


humset_v2 funnel_names: $2.36, malformed 0.0%; spent $8.51


humset_v2 funnel_cards: $3.74, malformed 0.0%; spent $12.25


humset_v2 walk_strict: $1.69, malformed 0.0%; spent $13.95


humset_v2 walk_branching: $2.04, malformed 0.0%; spent $15.99


... (2 outputs left out)

humset_v2 llm_ensemble tuning: $0.57, malformed 0.5%; spent $28.22


humset_v2 llm_ensemble: $36.62, malformed 0.4%; spent $64.84


humset_v2 ki_fsprompt tuning: $0.03, malformed 0.3%; spent $64.87


humset_v2 ki_fsprompt: $1.91, malformed 0.3%; spent $66.78
LLM calls: $66.78, saved replies included


## Score every method both ways

The tuning excerpts are notebook 15's 4,000 development excerpts. Each method is scored on the full test set
by exact labels and tier by tier, and by HumBERT's measure; its labels on the sample's excerpts are then scored
and checked against the saved sample scores.

In [4]:
def same_as_sample(name, data, sample, labels, saved):
    by_id = dict(zip([item.id for item in data.test], labels, strict=True))
    lf.check_reproduces(name, ex.evaluate(data, sample, [by_id[item.id] for item in sample]), saved)


def score_all(made, spent):
    test = data.test
    sample = ex.sample_humset(data, config["samples"]["humset"]["per_language"], seed)
    saved = SAMPLE_SCORES
    order = np.random.default_rng(seed).permutation(len(data.dev))
    raw = [data.dev[int(i)] for i in order[: plan["fit_items"] + plan["cut_items"]]]
    fit, cut = nt.with_all_tiers(raw[: plan["fit_items"]]), nt.with_all_tiers(raw[plan["fit_items"] :])
    tune = raw[plan["fit_items"] :]
    out = {"llm": {}, "ranked": {}, "combined": {}, "paper": {"llm": {}, "ranked": {}, "combined": {}}}

    for method, entry in made.items():
        labels = entry.pop("labels")
        same_as_sample(method, data, sample, labels, saved["llm"][method]["scores"])
        out["llm"][method] = {"family": "llm", "scores": ex.evaluate(data, test, labels), **entry}
        out["paper"]["llm"][method] = lf.paper_scores_of_labels(test, labels, data.taxonomy)

    dev = sg.load_sources(root, data.name, SOURCES, "dev")
    full = sg.load_sources(root, data.name, SOURCES, "test")
    for method in SOURCES:
        merged = {**dev[method], **full[method]}
        tune_rankings = lf.source_rankings(merged, tune, method)
        test_rankings = lf.source_rankings(merged, test, method)
        row = lf.score_ranked(data, tune, tune_rankings, test, test_rankings, limits, thresholds)
        labels = row.pop("labels")
        same_as_sample(method, data, sample, labels, saved["ranked"][method]["scores"])
        out["ranked"][method] = {"family": "ranked", **row}
        paper = lf.paper_scores(tune, tune_rankings, test, test_rankings, data.taxonomy)
        out["paper"]["ranked"][method] = paper
    # HumBERT's own trainer chooses each label's threshold on every development excerpt; the tables use the
    # 4,000 the combined method can use. For comparison, the recipe with thresholds chosen on every one.
    merged = {**dev["humbert_recipe"], **full["humbert_recipe"]}
    out["recipe_cut_on_all_dev"] = lf.paper_scores(
        data.dev,
        lf.source_rankings(merged, data.dev, "humbert_recipe"),
        test,
        lf.source_rankings(merged, test, "humbert_recipe"),
        data.taxonomy,
    )

    names = STACK
    graph = cb.LabelGraph(data.taxonomy, [sorted(bm.gold_with_parents(item)) for item in data.train])
    stack = nt.Stack(
        names,
        fit,
        cut,
        nt.expand_sources(sg.load_sources(root, data.name, names, "dev")),
        nt.expand_sources(sg.load_sources(root, data.name, names, "test")),
        graph,
        words,
        settings,
    )
    tune_rankings, test_rankings = lf.combined_rankings(stack, test)
    exact_labels = {label for item in data.train for label in item.labels}
    row = lf.score_ranked(
        data,
        tune,
        lf.restrict(tune_rankings, exact_labels),
        test,
        lf.restrict(test_rankings, exact_labels),
        limits,
        thresholds,
    )
    labels = row.pop("labels")
    same_as_sample("combined", data, sample, labels, saved["combined"]["combined"]["scores"])
    out["combined"]["combined"] = {"family": "combined", "sources": list(names), "priors": False, **row}
    paper = lf.paper_scores(tune, tune_rankings, test, test_rankings, data.taxonomy)
    for group, value in FULL_TEST_COMBINED.items():
        if abs(paper[group] - value) > SAME_WITHIN:
            raise ValueError(f"combined {group}: {paper[group]} is not the saved full-test {value}")
    out["paper"]["combined"]["combined"] = paper

    present = set().union(*(bm.gold_with_parents(item) for item in test))
    out["labels_not_in_test"] = {
        group: sorted(labels - present) for group, labels in bm.humset_categories(data.taxonomy).items()
    }
    out["gold"] = {
        "labels": sum(len(item.labels) for item in test),
        "by_tier": {
            str(t): n
            for t, n in sorted(
                pd.Series([data.taxonomy.tiers[label] for item in test for label in item.labels])
                .value_counts()
                .items()
            )
        },
    }
    out["sizes"] = {"test": len(test), "tune": len(tune), "dev": len(data.dev), "sample": len(sample)}
    out["llm_cost_usd"] = round(spent, 2)
    return out


out = score_all(made, spent)
print("each method's sample excerpts give its saved sample scores; the combined method, its full-test ones")
print("labels no test excerpt has:", {g: len(v) for g, v in out["labels_not_in_test"].items()})

each method's sample excerpts give its saved sample scores; the combined method, its full-test ones
labels no test excerpt has: {'Sectors': 0, 'Pillars 1D': 0, 'Subpillars 1D': 0, 'Pillars 2D': 0, 'Subpillars 2D': 1}


In [5]:
pd.DataFrame(
    {
        method: {
            "exact-label F1": out[family][method]["scores"]["exact"]["f1"],
            "macro F1": out["paper"][family][method]["Mean"],
        }
        for family in ("llm", "ranked", "combined")
        for method in out[family]
    }
).T.sort_values("macro F1", ascending=False).round(3)

,exact-label F1,macro F1
combined,0.716,0.663
classifier,0.682,0.639
humbert_recipe,0.693,0.629
knn,0.685,0.587
omikuji,0.668,0.569
ki_fsprompt,0.585,0.528
reranker,0.542,0.510
xr_transformer,0.722,0.506
embedding_classifier,0.627,0.469
gliner,0.502,0.464


In [6]:
ex.save_results("17_humset_full_test", {ex.HUMSET_V2: out})
print("saved")

saved
